# DecoupleNet LoveDA Semantic Segmentation — Google Colab

Bu notebook DecoupleNet modelini LoveDA veri seti ile Google Colab'da egitir.

**Veri kaynagi:** HuggingFace (`chloechia/loveda`)

## Adimlar
1. Ortam kontrolu (GPU, disk)
2. Repo klonlama
3. Bagimlilik kurulumu
4. Pre-trained agirlik indir
5. LoveDA veri setini indir + hazirla
6. Veri setini dogrula
7. (Opsiyonel) Google Drive bagla
8. Egitimi baslat
9. TensorBoard ile izle
10. Sonuclari kaydet

In [ ]:
# ===================== HUCRE 1: ORTAM KONTROLU =====================
import torch
import subprocess

print("=" * 50)
print("ORTAM KONTROLU")
print("=" * 50)

# GPU kontrolu
gpu_available = torch.cuda.is_available()
print(f"GPU mevcut: {gpu_available}")
if gpu_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("UYARI: GPU bulunamadi! Egitim cok yavas olacak.")
    print("Runtime -> Change runtime type -> T4 GPU secin")

# Disk alani
result = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
print(f"\nDisk alani:\n{result.stdout}")

# Python versiyonu
import sys
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ===================== HUCRE 2: REPO KLONLAMA =====================
import os

if not os.path.exists('/content/DecoupleNet'):
    !git clone https://github.com/myrisee/DecoupleNet.git /content/DecoupleNet
    print("Repo klonlandi!")
else:
    print("Repo zaten mevcut.")

# segmentation dizinine git
%cd /content/DecoupleNet/segmentation
print(f"Calisma dizini: {os.getcwd()}")

In [ ]:
# ===================== HUCRE 3: BAGIMLILIK KURULUMU =====================
print("Bagimliliklar kuruluyor...")

# Ana bagimliliklar
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Proje bagimliliklari
!pip install -q timm catalyst==20.09 pytorch-lightning==1.9.0
!pip install -q albumentations einops ttach pytorch-toolbelt
!pip install -q opencv-python-headless scipy matplotlib tqdm addict
!pip install -q antialiased-cnns

# HuggingFace indirme icin
!pip install -q huggingface_hub

print("Bagimliliklar kuruldu!")

# Versiyon kontrolu
import torch, timm, pytorch_lightning as pl
print(f"PyTorch: {torch.__version__}")
print(f"timm: {timm.__version__}")
print(f"Lightning: {pl.__version__}")

In [ ]:
# ===================== HUCRE 4: PRE-TRAINED AGIRLIK INDIR =====================
import os

weights_dir = '/content/DecoupleNet/segmentation/backbone_weights'
weights_path = os.path.join(weights_dir, 'DecoupleNet_D2.pth')
weights_url = 'https://github.com/lwCVer/DecoupleNet/releases/download/weights/DecoupleNet_D2.pth'

os.makedirs(weights_dir, exist_ok=True)

if not os.path.exists(weights_path):
    print(f"Agirlik indiriliyor: {weights_url}")
    !wget -q --show-progress -O "{weights_path}" "{weights_url}"
    file_size = os.path.getsize(weights_path) / (1024 * 1024)
    print(f"Indirildi: {file_size:.1f} MB")
else:
    file_size = os.path.getsize(weights_path) / (1024 * 1024)
    print(f"Agirlik dosyasi zaten mevcut: {file_size:.1f} MB")

# PyTorch 2.6+ güvenlik: weights_only=False gerekiyor
import torch
try:
    ckpt = torch.load(weights_path, map_location='cpu', weights_only=False)
    print(f"Agirlik dogrulandi! Keys: {len(ckpt)} parametre")
    if 'model' in ckpt:
        print(f"Model parametre sayisi: {len(ckpt['model'])}")
except Exception as e:
    print(f"HATA: Agirlik dosyasi gecersiz: {e}")

In [ ]:
# ===================== HUCRE 5: LOVEDA VERI SETINI INDIR + HAZIRLA =====================
# Bu islem ~5-10 dakika surer (4.67 GB indirilecek)

%cd /content/DecoupleNet/segmentation

import os

data_root = '/content/data/LoveDA'

# Eger veri zaten hazirsa atla
train_ready = os.path.exists(os.path.join(data_root, 'Train', 'masks_png_convert'))
val_ready = os.path.exists(os.path.join(data_root, 'Val', 'masks_png_convert'))

if train_ready and val_ready:
    print("Veri seti zaten hazir! Indirme atlaniyor.")
else:
    !python tools/prepare_loveda_colab.py --data-root {data_root}

print("\nHazir!")

In [ ]:
# ===================== HUCRE 6: VERI SETINI DOGRULA =====================
import os
import glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

data_root = '/content/data/LoveDA'

# Dosya sayilari
train_imgs = sorted(glob.glob(os.path.join(data_root, 'Train', 'images_png', '*.png')))
train_masks = sorted(glob.glob(os.path.join(data_root, 'Train', 'masks_png_convert', '*.png')))
val_imgs = sorted(glob.glob(os.path.join(data_root, 'Val', 'images_png', '*.png')))
val_masks = sorted(glob.glob(os.path.join(data_root, 'Val', 'masks_png_convert', '*.png')))

print("VERI SETI ISTATISTIKLERI")
print("=" * 40)
print(f"Train goruntuleri:  {len(train_imgs)}")
print(f"Train maskeleri:   {len(train_masks)}")
print(f"Val goruntuleri:    {len(val_imgs)}")
print(f"Val maskeleri:     {len(val_masks)}")
print(f"Toplam:            {len(train_imgs) + len(val_imgs)} goruntu")

# Ornek gorsellestirme
if len(train_imgs) > 0:
    img = Image.open(train_imgs[0]).convert('RGB')
    mask = Image.open(train_masks[0]).convert('L')
    mask_arr = np.array(mask)
    unique_vals = np.unique(mask_arr)
    print(f"\nOrnek goruntu boyutu: {img.size}")
    print(f"Maske deger araligi: {unique_vals}")

    CLASSES = ('background', 'building', 'road', 'water', 'barren', 'forest', 'agricultural')
    PALETTE = [[255,255,255], [255,0,0], [255,255,0], [0,0,255], [159,129,183], [0,255,0], [255,195,128]]

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(img)
    axes[0].set_title(f'Goruntu: {os.path.basename(train_imgs[0])}')
    axes[0].axis('off')

    # Maskeyi renklendir
    mask_rgb = np.zeros((*mask_arr.shape, 3), dtype=np.uint8)
    for cls_idx, color in enumerate(PALETTE):
        if cls_idx in unique_vals:
            mask_rgb[mask_arr == cls_idx] = color
    axes[1].imshow(mask_rgb)
    axes[1].set_title(f'Maske: {os.path.basename(train_masks[0])}')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    # Sinif dagilimi
    print("\nSinif dagilimi (ilk ornek maske):")
    for val in unique_vals:
        if val < len(CLASSES):
            count = np.sum(mask_arr == val)
            pct = count / mask_arr.size * 100
            print(f"  {CLASSES[val]:15s}: {count:>8d} piksel ({pct:.1f}%)")

In [ ]:
# ===================== HUCRE 7: (OPSIYONEL) GOOGLE DRIVE BAGLA =====================
# Model checkpoint'lerini kalici olarak kaydetmek icin Google Drive kullanin

from google.colab import drive
import shutil

DRIVE_MOUNT = '/content/drive'
CHECKPOINT_DRIVE_DIR = '/content/drive/MyDrive/DecoupleNet_checkpoints'

# Google Drive bagla
if not os.path.ismount(DRIVE_MOUNT):
    drive.mount(DRIVE_MOUNT)
    print("Google Drive baglandi!")
else:
    print("Google Drive zaten bagli.")

# Checkpoint dizinini olustur
os.makedirs(CHECKPOINT_DRIVE_DIR, exist_ok=True)

# Symbolic link olustur: model_weights/loveda -> Drive
local_weights = '/content/DecoupleNet/segmentation/model_weights/loveda'
os.makedirs(os.path.dirname(local_weights), exist_ok=True)

if os.path.islink(local_weights):
    os.unlink(local_weights)
elif os.path.exists(local_weights):
    shutil.rmtree(local_weights)

os.symlink(CHECKPOINT_DRIVE_DIR, local_weights)
print(f"Checkpoint dizini: {local_weights} -> {CHECKPOINT_DRIVE_DIR}")
print("Egitim bitince checkpoint'ler Google Drive'da kalici olacak!")

In [ ]:
# ===================== HUCRE 8: EGITIMI BASLAT =====================
# Tahmini sure: T4 ile ~12-15 saat (30 epoch)
# Colab ucretsiz hesap 12 saat zaman asimi var
# Her epoch sonunda checkpoint kaydedilir

%cd /content/DecoupleNet/segmentation

!python train_supervision.py -c config/loveda/train_decouplenet_colab.py

In [ ]:
# ===================== HUCRE 9: TENSORBOARD ILE IZLE =====================
# Egitim sirasinda bu hucreyi calistirarak metrikleri izleyebilirsiniz

%load_ext tensorboard
%tensorboard --logdir /content/DecoupleNet/segmentation/lightning_logs

In [ ]:
# ===================== HUCRE 10: SONUCLARI KAYDET =====================
import os
import shutil
import glob

print("EGITIM SONUCLARI")
print("=" * 50)

# Checkpoint dosyalari
ckpt_dir = '/content/DecoupleNet/segmentation/model_weights/loveda'
if os.path.exists(ckpt_dir):
    ckpt_files = glob.glob(os.path.join(ckpt_dir, '*.ckpt'))
    print(f"Checkpoint dosyalari ({len(ckpt_files)}):")
    for f in ckpt_files:
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f"  {os.path.basename(f):50s} {size_mb:.1f} MB")

# Google Drive'a kopyala (eger Drive bagliysa)
drive_ckpt_dir = '/content/drive/MyDrive/DecoupleNet_checkpoints'
if os.path.ismount('/content/drive') and os.path.exists(ckpt_dir):
    # Eger symbolic link degilse, manuel kopyala
    if not os.path.islink(ckpt_dir):
        os.makedirs(drive_ckpt_dir, exist_ok=True)
        for f in glob.glob(os.path.join(ckpt_dir, '*.ckpt')):
            dst = os.path.join(drive_ckpt_dir, os.path.basename(f))
            if not os.path.exists(dst):
                shutil.copy2(f, dst)
                print(f"  Kopyalandi: {os.path.basename(f)}")
        print("\nTum checkpoint'ler Google Drive'a kaydedildi!")
    else:
        print("\nCheckpoint'ler zaten Google Drive'a dogrudan kaydedildi (symlink).")

# Log dosyalari
log_dir = '/content/DecoupleNet/segmentation/lightning_logs'
if os.path.exists(log_dir):
    log_files = glob.glob(os.path.join(log_dir, '**', '*.csv'), recursive=True)
    print(f"\nLog dosyalari ({len(log_files)}):")
    for f in log_files:
        print(f"  {f}")

print("\nTamamlandi!")